# Exercise 21.4: Temperature, Top-$k$ & Top-$p$ Sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-21/04-sampling-temperature-topk.ipynb)

- **Book Reference:** Chapter 21 (*Mathematics of Transformers and Large Language Models*, Section 21.3)
- **Estimated Time:** 15–20 minutes
- **Learning Objectives:**
  1. Implement temperature scaling $\operatorname{softmax}(\mathbf{z} / \tau)$ and observe how $\tau \to 0$ sharpens vs. $\tau > 1$ flattens probabilities.
  2. Implement top-$k$ filtering and top-$p$ (nucleus) filtering on vocabulary logits.
  3. Sample next tokens deterministically with `np.random.default_rng(42)`.

In [ ]:
import numpy as np

## Mathematical Background

At the final step of an LLM forward pass, the model produces vocabulary logits $\mathbf{z} \in \mathbb{R}^V$.
1. **Temperature scaling** divides logits by $\tau > 0$: $\mathbf{p} = \operatorname{softmax}(\mathbf{z} / \tau)$.
   - Small $\tau$ (e.g. $0.2$) concentrates mass on the highest-logit token.
   - Large $\tau$ (e.g. $2.0$) makes the distribution flatter and more random.
2. **Top-$k$ filtering** keeps only the $k$ largest logits and sets all others to $-\infty$.
3. **Top-$p$ (nucleus) filtering** sorts probabilities descending and keeps the smallest prefix whose cumulative probability is $\ge p$, masking the tail to $-\infty$.

In [ ]:
def softmax_probs(logits: np.ndarray) -> np.ndarray:
    """Numerically stable 1D softmax."""
    # TODO: Compute stable softmax of 1D logits array
    raise NotImplementedError("Complete this TODO")


def apply_temperature(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Scale logits by 1 / temperature (temperature > 0)."""
    # TODO: Validate temperature > 0 and return logits / temperature
    raise NotImplementedError("Complete this TODO")


def top_k_filter(logits: np.ndarray, k: int) -> np.ndarray:
    """Mask all logits outside the top-k largest values to -np.inf."""
    # TODO: Keep top k values, set all others to -np.inf
    raise NotImplementedError("Complete this TODO")


def top_p_filter(logits: np.ndarray, p: float) -> np.ndarray:
    """Nucleus (top-p) filtering: keep smallest token set with cumulative prob >= p."""
    # TODO: Sort probabilities descending, find cutoff where cumsum >= p, mask tail logits to -np.inf
    raise NotImplementedError("Complete this TODO")


def sample_next_token(
    logits: np.ndarray,
    rng: np.random.Generator,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
) -> int:
    """Apply temperature, optional top-k / top-p filtering, and sample a token ID."""
    # TODO: Chain apply_temperature -> top_k_filter -> top_p_filter -> softmax_probs -> rng.choice
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- In `top_p_filter`, `cum_probs - sorted_probs >= p` identifies all tokens *after* the cumulative probability has already reached `p`, so the token that crosses `p` is still kept.
- Always keep at least the top-1 token (`remove_mask[0] = False`).
- Use `int(rng.choice(len(probs), p=probs))` to return a plain Python `int`.

</details>

In [ ]:
logits = np.array([1.0, 3.0, 0.5, -1.0, 2.0])  # argmax is index 1

# 1. Temperature effect: lower temp -> higher probability on argmax
p_low = softmax_probs(apply_temperature(logits, 0.2))
p_high = softmax_probs(apply_temperature(logits, 2.0))
assert p_low[1] > p_high[1], "Lower temperature must sharpen probability on top token"

# 2. Top-k=1 always selects argmax (index 1)
rng = np.random.default_rng(42)
for _ in range(10):
    assert sample_next_token(logits, rng, temperature=1.0, top_k=1) == 1

# 3. Top-k=2 masks 3 entries to -inf
f_k2 = top_k_filter(logits, k=2)
assert np.sum(np.isfinite(f_k2)) == 2

# 4. Top-p=0.7 keeps at least index 1 and 4 (probs ~ 0.60 and 0.22 at tau=1)
f_p = top_p_filter(logits, p=0.7)
assert np.isfinite(f_p[1]) and np.isfinite(f_p[4])
assert np.sum(np.isfinite(f_p)) == 2

tok = sample_next_token(logits, np.random.default_rng(42), temperature=0.8, top_k=3, top_p=0.9)
assert 0 <= tok < len(logits)
print(f"All checks passed! Sampled token={tok}, p_low[1]={p_low[1]:.4f}, p_high[1]={p_high[1]:.4f}")

## Think Deeper

1. Why is top-$p$ (nucleus) sampling often preferred over fixed top-$k$ sampling when an LLM's next-token distribution varies between very confident and very uncertain?
2. Which loss function is minimised during LLM pre-training, and which optimiser family updates the weights (Exercise 21.4)?